# Capa gold

In [7]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

from pyspark.sql import functions as F
from app.utils.spark import SparkClient
from app.utils.globals import globals

spark_client = SparkClient()
spark = spark_client.get_session()

GOLD_MARTS_PATH = "data/gold/marts"
GOLD_ML_PATH = "data/gold/ml"

## Resultados

### Marts de ML (dashboards predictivos)

#### Categorización de viajes (K-Modes)

Datos existentes:

In [17]:
df_fraude = spark.read.parquet(str(globals.project_root / GOLD_ML_PATH / "ml_feat_arima_trips"))
df_fraude.show()

+----------+-------------------+----------+-----------+---+----------+----------+---------+----+-----+
|service_id|        pickup_hour|trip_count|hour_of_day|dow|is_weekend|is_holiday|  borough|year|month|
+----------+-------------------+----------+-----------+---+----------+----------+---------+----+-----+
|     green|2025-10-23 12:00:00|        78|         12|  4|     false|     false|Manhattan|2025|   10|
|     green|2025-10-25 17:00:00|        60|         17|  6|      true|     false|Manhattan|2025|   10|
|     green|2025-10-12 09:00:00|        24|          9|  7|      true|     false|Manhattan|2025|   10|
|     green|2025-10-16 23:00:00|        15|         23|  4|     false|     false|Manhattan|2025|   10|
|     green|2025-10-05 14:00:00|        52|         14|  7|      true|     false|Manhattan|2025|   10|
|     green|2025-10-07 01:00:00|         1|          1|  2|     false|     false|Manhattan|2025|   10|
|     green|2025-10-16 05:00:00|         9|          5|  4|     false|   

#### Categorización de viajes (K-Modes)

Datos existentes:

In [16]:
df_fraude = spark.read.parquet(str(globals.project_root / GOLD_ML_PATH / "ml_feat_kmodes_trips"))
df_fraude.show()

+--------------------+--------+--------------+--------------+----------+----------+--------------+-------------+-----------------+---------+----------+----+-----+
|             trip_id|date_key|pu_location_id|do_location_id|borough_pu|borough_do|franja_horaria|dia_categoria|hvfhs_license_num|vendor_id|service_id|year|month|
+--------------------+--------+--------------+--------------+----------+----------+--------------+-------------+-----------------+---------+----------+----+-----+
|4d76c6e10af71df33...|20250201|           234|           141| Manhattan| Manhattan|     Madrugada|Fin de Semana|           HV0003|     NULL|     fhvhv|2025|    2|
|d6a05c285431d2e26...|20250201|            78|           215|     Bronx|    Queens|     Madrugada|Fin de Semana|           HV0003|     NULL|     fhvhv|2025|    2|
|7d8c7acf734c5d107...|20250201|            28|           130|    Queens|    Queens|     Madrugada|Fin de Semana|           HV0003|     NULL|     fhvhv|2025|    2|
|082b1befdb8b81e84...|

#### Detección de fraude (Isolation Forest)

Datos existentes:

In [15]:
df_fraude = spark.read.parquet(str(globals.project_root / GOLD_ML_PATH / "ml_feat_isolation_fraud"))
df_fraude.groupBy("ratecode_id").agg(F.avg("fare_amount"), F.avg("extra"), F.avg("mta_tax")).show()

+-----------+------------------+--------------------+--------------------+
|ratecode_id|  avg(fare_amount)|          avg(extra)|        avg(mta_tax)|
+-----------+------------------+--------------------+--------------------+
|          5| 69.96590444261454|  0.9920119211330769| 0.02083680047649496|
|          2| 69.97216233961447|  1.5547615772195438|  0.4998508319404193|
|          1| 16.52493850971394|   1.208972388980984|   0.500872566757254|
|          3| 75.52819967669124|  1.7056438984780653|0.004417559987186...|
|          6|107.27679245283021|  0.7405660377358491|  0.5660377358490566|
|         99| 34.58087365412611|0.001004595955012...| 0.49896233604300333|
|          4|114.07447175275001|  1.7442785204341809|  0.3545680957237561|
+-----------+------------------+--------------------+--------------------+

